# 310 — Classification data-prep & feature cache

Builds the classification-ready feature matrices **once** and caches them to disk so the
experiment notebooks (`320`, `330`) load instantly and never re-walk the ERSP tree.

**Source** — the same per-electrode ERSP `.npy` files the clustering pipeline uses:
`01_FBM_Analysis/outputs/04_ersp_LM_RAWONLY/<pid>/LM/ERSP_matrix/<cond>/`.

**Four feature variants — a nested, time-matched family** so "full spectrum vs
high-gamma" is a *fair* test (HG and full share a time grid; only frequency changes):
| variant | what | freq × time | dims |
|---|---|---|---|
| `full_300` | full spectrum, 15 bands | 15 × 300 | 4500 |
| `hg_300` | single 70–150 Hz line | 1 × 300 | 300 |
| `full_30` | full spectrum, downsampled time | 15 × 30 | 450 |
| `hg_30` | single HG line, downsampled time | 1 × 30 | 30 |

Valid contrasts are the **matched pairs**: `full_300 vs hg_300` and `full_30 vs hg_30`
(frequency content, time held fixed); `full_300 vs full_30` isolates time resolution.
⚠️ Frequency and time go hand in hand — the STFT couples their resolution, so even
matched grids aren't a perfectly clean separation (stated, not hidden).

**What it caches**
- **Condition task** (gated): one sample per high-activity electrode × condition.
- **Parcellation task**: one sample per electrode = `audio ⊕ picture ⊕ reading`
  concatenated, for each variant × {Yeo-7, Yeo-17}. An electrode is kept iff it has all
  three conditions, is high-activity in ≥1 condition, and sits in a real Yeo network
  (medial-wall / white-matter / unknown contacts are dropped).

Only patients with **all three** conditions enter either task. Run this first, then 320 & 330.


In [ ]:
import os, sys
from pathlib import Path
import numpy as np, pandas as pd
sys.path.insert(0, str(Path('..').resolve()))
from functions import lf_classify as C

# Server UNC path first, local relative path as fallback (mirrors 02's notebooks).
INPUT_DIR = Path(r'\\nasac-m2.unige.ch\m-HumanNeuronLab\ANALYSIS\FLM\Analysis_LoraFanda\01_FBM_Analysis\outputs\04_ersp_LM_RAWONLY')
if not INPUT_DIR.exists():
    INPUT_DIR = Path('../01_FBM_Analysis/outputs/04_ersp_LM_RAWONLY').resolve()

print('INPUT_DIR :', INPUT_DIR, '| exists:', INPUT_DIR.exists())
print('CACHE     :', C.DATASET_CACHE)
print('COORDS    :', C.COORDS_DIR, '| exists:', C.COORDS_DIR.exists())
print('variants  :', C.VARIANTS)


## 1 — Load the full ungated dataset
Walks every electrode × condition ERSP for patients that have all three conditions. The
high-activity flag is computed per sample (so both tasks derive from this one object).
The heavy walk is itself cached by `prepare_dataset`, so re-running is cheap.


In [ ]:
df_meta, X_3d = C.prepare_full_dataset(INPUT_DIR)
print('samples:', len(df_meta), '| X_3d:', X_3d.shape)
print('patients:', sorted(df_meta.patient_id.unique()))
df_meta.head()


## 2 — Condition task — cache the 4 feature variants
Gated electrode × condition samples, labelled by condition.


In [ ]:
for v in C.VARIANTS:
    X, y, groups, meta, cols = C.build_condition_arrays(df_meta, X_3d, v)
    d = C.save_arrays('condition', None, v, X, y, groups, meta, cols)
    print('  saved ->', d)


## 3 — Parcellation task — cache 4 variants × {Yeo-7, Yeo-17}
One sample per electrode; features = the three conditions concatenated in fixed order
`[audio, picture, reading]`. Label = the electrode's Yeo network.


In [ ]:
for n_net in (7, 17):
    for v in C.VARIANTS:
        X, y, groups, meta, cols = C.build_parcellation_arrays(
            df_meta, X_3d, v, n_networks=n_net)
        d = C.save_arrays('parcellation', f'yeo{n_net}', v, X, y, groups, meta, cols)
        print('  saved ->', d)


## 4 — Summary
The cache under `outputs/_dataset/classification/` now holds every (task × variant) matrix.
320 and 330 read these directly — no need to re-run 310 unless the upstream ERSPs or the
high-activity / Yeo filters change.


In [ ]:
rows = []
for v in C.VARIANTS:
    X, y, g, m, c = C.load_arrays('condition', None, v)
    rows.append(('condition', '-', v, str(X.shape), len(set(y)), len(set(g))))
for n_net in (7, 17):
    for v in C.VARIANTS:
        X, y, g, m, c = C.load_arrays('parcellation', f'yeo{n_net}', v)
        rows.append(('parcellation', f'yeo{n_net}', v, str(X.shape), len(set(y)), len(set(g))))
pd.DataFrame(rows, columns=['task', 'target', 'variant', 'X.shape', 'n_classes', 'n_patients'])
